# 🧠 AGSE-VNet Clinical Brain Tumor Analysis
### Patient-Facing Segmentation, Atlas Mapping & Report Generation
**Model:** AGSE-VNet Epoch-55 (3D U-Net + Attention Gates + SE Blocks)  
**Pipeline:** Patient MRI Upload → Segmentation → MNI152 Registration → Julich Atlas Mapping → AI Report  
**Reference:** Valerio et al., 2025 — *"From segmentation to explanation: Generating textual reports from MRI with LLMs"*

---
> **Instructions:** Run all cells top-to-bottom. Cells marked `⚙️ USER INPUT` require you to provide data.

| Label | Sub-region | BraTS Convention |
|---|---|---|
| 1 | NCR – Necrotic Core | Tumor Core (TC) |
| 2 | ED – Peritumoral Edema | Peritumoral Edema |
| 3 | ET – Enhancing Tumor | GD-Enhancing Tumor |

> **TC (Tumor Core) = NCR (1) + ET (3)** — BraTS / paper convention.


## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'nibabel', 'monai', 'einops', 'scikit-learn', 'tqdm',
    'antspyx', 'nilearn', 'siibra',
    'huggingface_hub', 'transformers', 'accelerate', 'requests',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ All packages installed.')


## Cell 2 — Imports & Global Config

In [ ]:
import os, gc, json, random, time, base64, warnings, io
warnings.filterwarnings('ignore')

import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, HTML

import torch, torch.nn as nn, torch.nn.functional as F
from torch.cuda.amp import autocast
import ants, siibra
from nilearn import datasets, image as nlimage

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU — inference will be slow.')

CFG = dict(
    modalities      = ['flair', 't1ce', 't2', 't1'],
    num_classes     = 4,
    patch_size      = (96, 96, 96),
    base_filters    = 16,
    se_reduction    = 4,
    dropout         = 0.1,
    mixed_precision = True,
)
LABEL    = dict(BACKGROUND=0, NCR=1, EDEMA=2, ET=3)
TC_LABELS = {LABEL['NCR'], LABEL['ET']}
WT_LABELS = {LABEL['NCR'], LABEL['EDEMA'], LABEL['ET']}
ET_LABELS = {LABEL['ET']}
COLOR_MAP = {'Tumor_Core':'red','Peritumoral_Edema':'yellow','GD_Enhancing_Tumor':'green'}
print('✅ Config ready.')


## Cell 3 — Mount Google Drive & Locate Checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive')
CHECKPOINT_DIR = DRIVE_ROOT / 'AGSE_UNet3D_Models'
OUTPUT_DIR     = DRIVE_ROOT / 'AGSE_PatientReports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_BEST = CHECKPOINT_DIR / 'best_model.pth'
assert CKPT_BEST.exists(), (
    f'Checkpoint not found: {CKPT_BEST}'
    f'  Place best_model.pth in MyDrive/AGSE_UNet3D_Models/'
)
print(f'✅ Checkpoint : {CKPT_BEST}')
print(f'   Output     : {OUTPUT_DIR}')


## Cell 4 — ⚙️ USER INPUT: Patient Details Form

Run this cell — an **interactive widget form** will appear. Fill in all fields and click **✅ Save Patient Details** before Cell 5.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ─────────────────────────────────────────────────────────────────────────────
# Interactive Patient Information Form
# Fill in each field and click "Save Patient Details" before running Cell 5.
# ─────────────────────────────────────────────────────────────────────────────

header_html = widgets.HTML("""
<div style="background:linear-gradient(135deg,#1a237e,#4a148c);
            color:white;padding:20px 24px;border-radius:10px;margin-bottom:16px">
  <h2 style="margin:0;font-size:20px">🏥 Patient Information Form</h2>
  <p style="margin:6px 0 0;opacity:.85;font-size:13px">
    Fill in all fields and click Save before continuing to Cell 5.
  </p>
</div>""")

style = {'description_width': '160px'}
lay_w = widgets.Layout(width='480px')
lay_l = widgets.Layout(width='600px')
lay_a = widgets.Layout(width='600px', height='110px')

w_id   = widgets.Text(
    value='PT-001', description='Patient ID *',
    placeholder='e.g. PT-001 or BraTS2021_00045',
    style=style, layout=lay_w)

w_age  = widgets.BoundedIntText(
    value=45, min=1, max=120, description='Age (years) *',
    style=style, layout=widgets.Layout(width='320px'))

w_sex  = widgets.Dropdown(
    options=['Male', 'Female', 'Other / Not specified'],
    value='Male', description='Biological Sex *',
    style=style, layout=widgets.Layout(width='380px'))

w_ref  = widgets.Text(
    value='Dr. A. Sharma', description='Referring Physician',
    placeholder='Full name and department',
    style=style, layout=lay_l)

w_inst = widgets.Text(
    value='', description='Institution',
    placeholder='Hospital / Clinic name (optional)',
    style=style, layout=lay_l)

w_diag = widgets.Dropdown(
    options=[
        'Suspected Glioma',
        'Suspected Glioblastoma (GBM)',
        'Known Glioma – Follow-up',
        'Post-operative Assessment',
        'Radiation Necrosis vs Recurrence',
        'Metastasis – Rule out Primary',
        'Other / Unknown',
    ],
    value='Suspected Glioma',
    description='Clinical Indication *',
    style=style, layout=lay_l)

w_sym  = widgets.SelectMultiple(
    options=[
        'Headache', 'Seizures', 'Cognitive changes',
        'Motor deficits', 'Speech difficulties', 'Visual disturbances',
        'Nausea / Vomiting', 'Fatigue', 'Personality changes', 'Asymptomatic',
    ],
    value=['Headache'],
    description='Presenting Symptoms',
    style=style,
    layout=widgets.Layout(width='480px', height='160px'))

w_notes = widgets.Textarea(
    value='Patient presents with recurrent headache and mild cognitive changes. '
          'MRI ordered to rule out intracranial mass.',
    description='Clinical Notes *',
    placeholder='Free-text clinical history and relevant findings...',
    style=style, layout=lay_a)

w_kps  = widgets.IntSlider(
    value=80, min=10, max=100, step=10,
    description='KPS Score',
    style=style, layout=widgets.Layout(width='480px'),
    readout=True,
    continuous_update=False)

w_kps_label = widgets.HTML(
    '<span style="font-size:12px;color:#718096">'
    'Karnofsky Performance Status (10=worst, 100=best functioning)</span>')

w_btn  = widgets.Button(
    description='✅  Save Patient Details',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='260px', height='40px', margin='20px 0 8px'))

w_status = widgets.Output()

PATIENT_META = {}
PATIENT_ID   = 'PT-001'

def on_save(b):
    global PATIENT_META, PATIENT_ID
    with w_status:
        clear_output()
        pid = w_id.value.strip() or 'PT-001'
        PATIENT_ID = pid
        symptoms   = ', '.join(w_sym.value) if w_sym.value else 'Not specified'
        PATIENT_META = {
            'patient_id'          : pid,
            'age'                 : w_age.value,
            'sex'                 : w_sex.value.replace(' / Not specified',''),
            'referring_physician' : w_ref.value.strip() or 'Not provided',
            'institution'         : w_inst.value.strip() or 'Not provided',
            'clinical_indication' : w_diag.value,
            'presenting_symptoms' : symptoms,
            'kps_score'           : w_kps.value,
            'clinical_notes'      : w_notes.value.strip(),
            'scan_date'           : time.strftime('%Y-%m-%d'),
        }
        print('\n' + '─'*60)
        print('  ✅  PATIENT RECORD SAVED')
        print('─'*60)
        for k, v in PATIENT_META.items():
            print(f'  {k:<26}: {v}')
        print('─'*60)
        print('\n  ▶  Continue to Cell 5 — Upload MRI Files')

w_btn.on_click(on_save)

form = widgets.VBox([
    header_html,
    widgets.HTML('<b style="color:#2d3748">Basic Information</b>'),
    w_id, w_age, w_sex,
    widgets.HTML('<b style="color:#2d3748;margin-top:12px">Clinical Information</b>'),
    w_ref, w_inst, w_diag,
    widgets.HTML('<b style="color:#2d3748">Presenting Symptoms</b> '
                 '<span style="font-size:12px;color:#718096">(Ctrl+click to select multiple)</span>'),
    w_sym,
    widgets.HTML('<b style="color:#2d3748">Performance Status</b>'),
    widgets.VBox([w_kps, w_kps_label]),
    widgets.HTML('<b style="color:#2d3748">Clinical Notes</b>'),
    w_notes,
    w_btn,
    w_status,
], layout=widgets.Layout(padding='8px', gap='8px'))

display(form)


## Cell 5 — ⚙️ USER INPUT: Upload Patient MRI Files

### Option A (Upload): Directly upload 4 NIfTI files
- `*_flair.nii.gz`, `*_t1ce.nii.gz`, `*_t2.nii.gz`, `*_t1.nii.gz`

### Option B (Drive folder): Set `USE_DRIVE_FOLDER = True` and configure path


In [ ]:
from google.colab import files as colab_files

# ─── Choose method ────────────────────────────────────────────────────────
USE_DRIVE_FOLDER     = False
DRIVE_PATIENT_FOLDER = DRIVE_ROOT / 'BraTS2021_Dataset' / 'BraTS2021_00045'
# ─────────────────────────────────────────────────────────────────────────

UPLOAD_DIR = Path('/content/patient_mri')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
MODALITY_PATHS = {}

if USE_DRIVE_FOLDER:
    pat_dir  = Path(DRIVE_PATIENT_FOLDER)
    pat_stem = pat_dir.name
    for mod in CFG['modalities']:
        cand = pat_dir / f'{pat_stem}_{mod}.nii.gz'
        assert cand.exists(), f'Missing: {cand}'
        MODALITY_PATHS[mod] = str(cand)
    print(f'Loaded from Drive folder: {pat_dir}')
else:
    print('Upload your 4 NIfTI files (flair, t1ce, t2, t1):')
    uploaded = colab_files.upload()
    mod_keywords = {
        'flair': ['flair'],
        't1ce' : ['t1ce','t1c','ce'],
        't2'   : ['_t2','t2w'],
        't1'   : ['_t1.','_t1_'],
    }
    for fname in uploaded:
        dest = UPLOAD_DIR / fname
        dest.write_bytes(uploaded[fname])
        fname_lower = fname.lower()
        for mod, kws in mod_keywords.items():
            if any(k in fname_lower for k in kws) and mod not in MODALITY_PATHS:
                MODALITY_PATHS[mod] = str(dest)
                print(f'  {fname}  ->  [{mod}]')
                break
    if len(MODALITY_PATHS) < 4:
        print('Auto-detect incomplete; mapping by order: flair,t1ce,t2,t1')
        for mod, fname in zip(CFG['modalities'], list(uploaded.keys())):
            if mod not in MODALITY_PATHS:
                MODALITY_PATHS[mod] = str(UPLOAD_DIR / fname)

assert len(MODALITY_PATHS) == 4, f'Need 4 modalities, got {len(MODALITY_PATHS)}'
print('\nMRI files ready:')
for mod, path in MODALITY_PATHS.items():
    sz = Path(path).stat().st_size / 1e6
    print(f'  [{mod}]  {Path(path).name}  ({sz:.1f} MB)')


## Cell 6 — AGSE-VNet Architecture (identical to training)

In [ ]:
class ConvBnReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel, stride, padding, bias=False),
            nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        B, C = x.shape[:2]
        return x * self.fc(x.view(B,C,-1).mean(-1)).view(B,C,1,1,1)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, se_reduction=4, dropout=0.1):
        super().__init__()
        self.conv1    = ConvBnReLU(in_ch, out_ch)
        self.conv2    = nn.Sequential(
            nn.Conv3d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm3d(out_ch))
        self.se       = SEBlock3D(out_ch, se_reduction)
        self.drop     = nn.Dropout3d(dropout)
        self.relu     = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv3d(in_ch,out_ch,1,bias=False),
                          nn.BatchNorm3d(out_ch)) if in_ch!=out_ch else nn.Identity())
    def forward(self, x):
        return self.relu(self.se(self.drop(self.conv2(self.conv1(x)))) + self.shortcut(x))

class AttentionGate3D(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv3d(F_g,F_int,1), nn.BatchNorm3d(F_int))
        self.W_x  = nn.Sequential(nn.Conv3d(F_l,F_int,1), nn.BatchNorm3d(F_int))
        self.psi  = nn.Sequential(nn.Conv3d(F_int,1,1), nn.BatchNorm3d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g); x1 = self.W_x(x)
        if g1.shape != x1.shape:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='trilinear', align_corners=False)
        return x * self.psi(self.relu(g1 + x1))

class UNet3D_AGSE(nn.Module):
    def __init__(self, in_ch=4, num_classes=4, base_filters=16, se_r=4, dropout=0.1):
        super().__init__()
        F = base_filters
        self.enc1=ResidualBlock(in_ch,F,se_r,dropout)
        self.enc2=ResidualBlock(F,F*2,se_r,dropout)
        self.enc3=ResidualBlock(F*2,F*4,se_r,dropout)
        self.enc4=ResidualBlock(F*4,F*8,se_r,dropout)
        self.down1=nn.Conv3d(F,F,2,stride=2,bias=False)
        self.down2=nn.Conv3d(F*2,F*2,2,stride=2,bias=False)
        self.down3=nn.Conv3d(F*4,F*4,2,stride=2,bias=False)
        self.down4=nn.Conv3d(F*8,F*8,2,stride=2,bias=False)
        self.bottleneck=ResidualBlock(F*8,F*16,se_r,dropout)
        self.ag4=AttentionGate3D(F*8,F*8,F*4)
        self.ag3=AttentionGate3D(F*4,F*4,F*2)
        self.ag2=AttentionGate3D(F*2,F*2,F)
        self.ag1=AttentionGate3D(F,F,F//2)
        self.up4=nn.ConvTranspose3d(F*16,F*8,2,stride=2)
        self.dec4=ResidualBlock(F*16,F*8,se_r,dropout)
        self.up3=nn.ConvTranspose3d(F*8,F*4,2,stride=2)
        self.dec3=ResidualBlock(F*8,F*4,se_r,dropout)
        self.up2=nn.ConvTranspose3d(F*4,F*2,2,stride=2)
        self.dec2=ResidualBlock(F*4,F*2,se_r,dropout)
        self.up1=nn.ConvTranspose3d(F*2,F,2,stride=2)
        self.dec1=ResidualBlock(F*2,F,se_r,dropout)
        self.out=nn.Conv3d(F,num_classes,1)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,nn.Conv3d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm3d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.down1(e1))
        e3=self.enc3(self.down2(e2)); e4=self.enc4(self.down3(e3))
        b=self.bottleneck(self.down4(e4))
        d4=self.dec4(torch.cat([self.up4(b),  self.ag4(self.up4(b),e4)],1))
        d3=self.dec3(torch.cat([self.up3(d4), self.ag3(self.up3(d4),e3)],1))
        d2=self.dec2(torch.cat([self.up2(d3), self.ag2(self.up2(d3),e2)],1))
        d1=self.dec1(torch.cat([self.up1(d2), self.ag1(self.up1(d2),e1)],1))
        return self.out(d1)

print(f'Architecture defined.')


## Cell 7 — Load Epoch-55 Checkpoint

In [ ]:
model = UNet3D_AGSE(
    in_ch=len(CFG['modalities']), num_classes=CFG['num_classes'],
    base_filters=CFG['base_filters'], se_r=CFG['se_reduction'], dropout=CFG['dropout']
).to(DEVICE)

ckpt = torch.load(CKPT_BEST, map_location=DEVICE)
if 'state_dict' in ckpt:         model.load_state_dict(ckpt['state_dict'])
elif 'model_state_dict' in ckpt: model.load_state_dict(ckpt['model_state_dict'])
elif 'model' in ckpt:            model.load_state_dict(ckpt['model'])
else:                            model.load_state_dict(ckpt)

model.eval()
saved_epoch  = ckpt.get('epoch', 54)
best_dice_wt = ckpt.get('best', 0.0)
print('=' * 52)
print(f'  Epoch       : {int(saved_epoch)+1}')
print(f'  Best Dice-WT: {best_dice_wt:.4f}')
print(f'  Parameters  : {sum(p.numel() for p in model.parameters()):,}')
print('=' * 52)


## Cell 8 — Load & Preprocess Patient MRI

In [ ]:
def znorm(vol):
    mask = vol > 0
    if not mask.any(): return vol
    vol = (vol - vol[mask].mean()) / (vol[mask].std() + 1e-8)
    return vol * mask

print(f'Loading MRI for patient: {PATIENT_ID}')
raw_nibs = {}
for mod in CFG['modalities']:
    raw_nibs[mod] = nib.load(MODALITY_PATHS[mod])
    print(f'  [{mod}]  shape={raw_nibs[mod].shape}')

NATIVE_AFFINE = raw_nibs['flair'].affine
NATIVE_HEADER = raw_nibs['flair'].header
SPACING       = np.sqrt((NATIVE_AFFINE[:3,:3]**2).sum(0))
VOX_VOL_MM3   = float(np.prod(SPACING))

volume = np.stack(
    [znorm(raw_nibs[m].get_fdata(dtype=np.float32)) for m in CFG['modalities']],
    axis=0  # (4, D, H, W)
)
print(f'\nVolume shape   : {volume.shape}')
print(f'Voxel spacing  : {np.round(SPACING,2)} mm')
print(f'Voxel vol      : {VOX_VOL_MM3:.3f} mm3')


## Cell 9 — Gaussian Sliding-Window Inference

In [ ]:
def sliding_window_inference(model, volume, patch=(96,96,96), stride=48, device=DEVICE):
    _, D, H, W = volume.shape
    pd, ph, pw = patch
    def g1d(n): x=np.linspace(-1,1,n); return np.exp(-2*x**2)
    gauss = torch.tensor(
        np.einsum('i,j,k->ijk', g1d(pd), g1d(ph), g1d(pw)).astype(np.float32)
    ).to(device)
    acc   = torch.zeros(CFG['num_classes'], D, H, W, device=device)
    count = torch.zeros(D, H, W, device=device)
    model.eval()
    with torch.no_grad():
        for d0 in range(0, max(D-pd+1,1), stride):
            d0 = min(d0, max(D-pd,0))
            for h0 in range(0, max(H-ph+1,1), stride):
                h0 = min(h0, max(H-ph,0))
                for w0 in range(0, max(W-pw+1,1), stride):
                    w0 = min(w0, max(W-pw,0))
                    pat = volume[:, d0:d0+pd, h0:h0+ph, w0:w0+pw]
                    dp,hp,wp = pat.shape[1:]
                    if dp<pd or hp<ph or wp<pw:
                        pat = np.pad(pat,[(0,0),(0,pd-dp),(0,ph-hp),(0,pw-wp)])
                    t = torch.tensor(pat[None], dtype=torch.float32).to(device)
                    with autocast(enabled=CFG['mixed_precision'] and device.type=='cuda'):
                        prob = torch.softmax(model(t)[0], dim=0)
                    acc[:, d0:d0+pd, h0:h0+ph, w0:w0+pw] += prob * gauss
                    count[   d0:d0+pd, h0:h0+ph, w0:w0+pw] += gauss
    return (acc / count.unsqueeze(0).clamp(1e-5)).cpu().numpy()

print(f'Running inference on patient {PATIENT_ID} ...')
t0 = time.time()
prob_map = sliding_window_inference(model, volume)
seg_pred = prob_map.argmax(0).astype(np.int32)
elapsed  = time.time() - t0

wt_vol_mm3 = float(np.isin(seg_pred, list(WT_LABELS)).sum()) * VOX_VOL_MM3
tc_vol_mm3 = float(np.isin(seg_pred, list(TC_LABELS)).sum()) * VOX_VOL_MM3
et_vol_mm3 = float(np.isin(seg_pred, list(ET_LABELS)).sum()) * VOX_VOL_MM3

print(f'\n✅ Inference done in {elapsed:.0f}s')
print(f'   Labels       : {np.unique(seg_pred)}')
print(f'   WT volume    : {wt_vol_mm3/1000:.2f} mm3')
print(f'   TC volume    : {tc_vol_mm3/1000:.2f} mm3')
print(f'   ET volume    : {et_vol_mm3/1000:.2f} mm3')

del prob_map; gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()


## Cell 10 — Save Native-Space Segmentation Masks

In [ ]:
nib.save(nib.Nifti1Image(seg_pred.astype(np.int16), NATIVE_AFFINE, NATIVE_HEADER),
         '/tmp/seg_pred_native.nii.gz')

tc_mask_native = np.isin(seg_pred, list(TC_LABELS)).astype(np.int16)
ed_mask_native = (seg_pred == LABEL['EDEMA']).astype(np.int16)
et_mask_native = (seg_pred == LABEL['ET']).astype(np.int16)

nib.save(nib.Nifti1Image(tc_mask_native, NATIVE_AFFINE, NATIVE_HEADER), '/tmp/tc_mask_native.nii.gz')
nib.save(nib.Nifti1Image(ed_mask_native, NATIVE_AFFINE, NATIVE_HEADER), '/tmp/ed_mask_native.nii.gz')
nib.save(nib.Nifti1Image(et_mask_native, NATIVE_AFFINE, NATIVE_HEADER), '/tmp/et_mask_native.nii.gz')

FLAIR_PATH = MODALITY_PATHS['flair']
print('✅ Native masks saved.')
print(f'   TC voxels : {int(tc_mask_native.sum()):,}')
print(f'   ED voxels : {int(ed_mask_native.sum()):,}')
print(f'   ET voxels : {int(et_mask_native.sum()):,}')


## Cell 11 — Register FLAIR → MNI152 Space (ANTsPy SyN)
*Takes 3–8 minutes. Do not interrupt.*


In [ ]:
print('Fetching MNI152 T1 template ...')
mni_dataset = datasets.fetch_icbm152_2009()
MNI_PATH    = mni_dataset.t1

fixed_img  = ants.image_read(MNI_PATH)
moving_img = ants.image_read(FLAIR_PATH)
print(f'Fixed  (MNI152)  : {fixed_img.shape}')
print(f'Moving (patient) : {moving_img.shape}')

print('Running SyN registration ...')
t_reg = time.time()
reg = ants.registration(
    fixed=fixed_img, moving=moving_img,
    type_of_transform='SyN',
    grad_step=0.1, flow_sigma=3, total_sigma=0, verbose=False,
)
FWD_TRANSFORMS = reg['fwdtransforms']
print(f'✅ Registration done in {time.time()-t_reg:.0f}s')

def warp_mask(src, transforms, fixed, out_path):
    warped = ants.apply_transforms(fixed=fixed, moving=ants.image_read(src),
                                   transformlist=transforms, interpolator='nearestNeighbor')
    ants.image_write(warped, out_path)
    return warped.numpy().astype(np.int32)

print('Warping masks ...')
tc_mni  = warp_mask('/tmp/tc_mask_native.nii.gz', FWD_TRANSFORMS, fixed_img, '/content/tc_mask_mni.nii.gz')
ed_mni  = warp_mask('/tmp/ed_mask_native.nii.gz', FWD_TRANSFORMS, fixed_img, '/content/ed_mask_mni.nii.gz')
et_mni  = warp_mask('/tmp/et_mask_native.nii.gz', FWD_TRANSFORMS, fixed_img, '/content/et_mask_mni.nii.gz')
seg_mni = warp_mask('/tmp/seg_pred_native.nii.gz', FWD_TRANSFORMS, fixed_img, '/content/seg_pred_mni.nii.gz')
ants.image_write(reg['warpedmovout'], '/content/flair_mni.nii.gz')

print(f'TC voxels in MNI : {int(tc_mni.sum()):,}')
print(f'ED voxels in MNI : {int(ed_mni.sum()):,}')
print(f'ET voxels in MNI : {int(et_mni.sum()):,}')


## Cell 12 — Julich Brain Atlas Mapping (siibra)

In [ ]:
from nilearn.image import resample_img

print('Initialising Julich Brain Atlas ...')
atlas        = siibra.atlases['human']
parcellation = atlas.get_parcellation('julich brain')
space_mni    = atlas.get_space('mni152')

print('Fetching labelled volume in MNI152 space ...')
pmap          = parcellation.get_map(space=space_mni, maptype='labelled')
atlas_vol_nib = pmap.fetch()
atlas_data    = atlas_vol_nib.get_fdata(dtype=np.float32).astype(np.int32)
atlas_affine  = atlas_vol_nib.affine
print(f'Atlas shape      : {atlas_data.shape}')
print(f'Unique region IDs: {len(np.unique(atlas_data[atlas_data>0]))}')

# Build label-to-name lookup
label_to_name = {}
if hasattr(pmap, 'regions') and pmap.regions:
    regions_src = pmap.regions
    if isinstance(regions_src, dict):
        for lbl, region in regions_src.items():
            label_to_name[int(lbl)] = region.name
    else:
        for region in regions_src:
            idx = getattr(region, 'index', None)
            if idx is not None:
                lbl = getattr(idx, 'label', None)
                if lbl is not None:
                    label_to_name[int(lbl)] = region.name
if not label_to_name:
    for uid in np.unique(atlas_data[atlas_data > 0]):
        label_to_name[int(uid)] = f'Julich_Region_{int(uid)}'
print(f'Region names loaded: {len(label_to_name)}')


## Cell 13 — Compute Regional Statistics

In [ ]:
tc_nib_mni  = nib.load('/content/tc_mask_mni.nii.gz')
tc_resampled = resample_img(
    tc_nib_mni,
    target_affine=atlas_affine, target_shape=atlas_data.shape, interpolation='nearest')
tc_in_atlas = tc_resampled.get_fdata(dtype=np.float32).astype(np.int32)

tc_voxel_coords = np.argwhere(tc_in_atlas > 0)
TC_TOTAL_VOXELS = len(tc_voxel_coords)
assert TC_TOTAL_VOXELS > 0, 'No Tumor Core voxels. Check segmentation / registration.'

region_voxel_count = {}
for coord in tc_voxel_coords:
    aid = int(atlas_data[coord[0], coord[1], coord[2]])
    if aid == 0: continue
    region_voxel_count[aid] = region_voxel_count.get(aid, 0) + 1

unique_ids, cnts = np.unique(atlas_data[atlas_data > 0], return_counts=True)
region_sizes = {int(a): int(c) for a, c in zip(unique_ids, cnts)}

records = []
for atlas_id, tc_count in region_voxel_count.items():
    region_name = label_to_name.get(atlas_id, f'Region_{atlas_id}')
    region_size = region_sizes.get(atlas_id, 1)
    records.append({
        'atlas_id'                      : atlas_id,
        'Region'                        : region_name,
        'tc_voxels'                     : tc_count,
        'region_size'                   : region_size,
        'Percentage_of_Tumor'           : round(100.0 * tc_count / TC_TOTAL_VOXELS, 2),
        'Percentage_of_Region_Affected' : round(100.0 * tc_count / region_size, 2),
    })
records.sort(key=lambda r: r['Percentage_of_Tumor'], reverse=True)
top5 = records[:5]

print('='*76)
print('  TOP-5 JULICH REGIONS — TUMOR CORE')
print('='*76)
print(f'  {"#":<3} {"Region":<42} {"% Tumor":>9}  {"% Region":>10}')
print('-'*76)
for i, r in enumerate(top5, 1):
    print(f'  {i:<3} {r["Region"]:<42} {r["Percentage_of_Tumor"]:>8.2f}%  {r["Percentage_of_Region_Affected"]:>9.2f}%')


## Cell 14 — Build Structured JSON (Paper Listing 1 Format)

In [ ]:
atlas_json = {
    'MRI_Scan': {
        'Patient'      : PATIENT_META,
        'Epoch_Loaded' : int(saved_epoch) + 1,
        'Tumor_Details': {
            'Spatial_Distribution': [
                {
                    'Region'                        : r['Region'],
                    'Percentage_of_Tumor'           : f'{r["Percentage_of_Tumor"]:.2f}%',
                    'Percentage_of_Region_Affected' : f'{r["Percentage_of_Region_Affected"]:.2f}%',
                    'TC_Voxels'                     : r['tc_voxels'],
                } for r in top5
            ],
            'Total_TC_Voxels'  : TC_TOTAL_VOXELS,
            'Regions_Affected' : len(region_voxel_count),
            'Volume_Estimates_cm3': {
                'Whole_Tumor'    : round(wt_vol_mm3 / 1000, 3),
                'Tumor_Core'     : round(tc_vol_mm3 / 1000, 3),
                'Enhancing_Tumor': round(et_vol_mm3 / 1000, 3),
            },
            'Semantic_Segmentation': {
                'Tumor_Core'        : {'Color': 'red',    'Labels': 'NCR(1)+ET(3)'},
                'Peritumoral_Edema' : {'Color': 'yellow', 'Labels': 'Edema(2)'},
                'GD_Enhancing_Tumor': {'Color': 'green',  'Labels': 'ET(3)'},
            },
            'Model_Used'  : 'AGSE-VNet Epoch-55',
            'Atlas_Used'  : 'Julich Brain Atlas (MNI152)',
        }
    }
}

json_path = f'/content/{PATIENT_ID}_atlas_mapping.json'
with open(json_path, 'w') as f:
    json.dump(atlas_json, f, indent=2)
print(json.dumps(atlas_json, indent=2)[:800], '...')
print(f'\n✅ JSON saved -> {json_path}')


## Cell 15 — MRI Visualization + Multi-Slice Export

Generates 6-panel static figure **and** exports 24 axial slice pairs (FLAIR / FLAIR+Seg) as base64 images for the interactive navigator in Cell 18.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 15 — MRI Visualization + Multi-Slice Export for Dashboard
# ─────────────────────────────────────────────────────────────────────────────
flair_d = nib.load('/content/flair_mni.nii.gz').get_fdata(dtype=np.float32)
seg_d   = nib.load('/content/seg_pred_mni.nii.gz').get_fdata(dtype=np.float32).astype(np.int32)
tc_d    = nib.load('/content/tc_mask_mni.nii.gz').get_fdata(dtype=np.float32)

centre = np.argwhere(tc_d > 0).mean(0).astype(int) if tc_d.sum() > 0 else np.array(flair_d.shape)//2

LABEL_CMAP = {1:[1,.2,.2], 2:[1,1,0], 3:[.2,1,.2]}
def seg_rgba(sl):
    h,w = sl.shape
    rgba = np.zeros((h,w,4), dtype=np.float32)
    for lbl, col in LABEL_CMAP.items():
        m = sl==lbl; rgba[m,:3]=col; rgba[m,3]=0.6
    return rgba

def slice_to_b64(flair_sl, seg_sl=None, dpi=80):
    fig, ax = plt.subplots(figsize=(3.2,3.2))
    fig.patch.set_facecolor('#0d0d0d')
    ax.set_facecolor('#0d0d0d')
    ax.imshow(np.rot90(flair_sl), cmap='gray', aspect='equal')
    if seg_sl is not None:
        ax.imshow(np.rot90(seg_rgba(seg_sl)), aspect='equal')
    ax.axis('off')
    plt.tight_layout(pad=0)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=dpi, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# ── Static 6-panel summary figure ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(
    f'AGSE-VNet Ep-55  |  Patient: {PATIENT_ID}  |  Julich Brain Atlas',
    fontsize=13, color='white', fontweight='bold', y=0.99)

planes = [
    ('Axial',    flair_d[:,:,centre[2]], seg_d[:,:,centre[2]]),
    ('Coronal',  flair_d[:,centre[1],:], seg_d[:,centre[1],:]),
    ('Sagittal', flair_d[centre[0],:,:], seg_d[centre[0],:,:]),
]
for col, (title, flair_sl, seg_sl) in enumerate(planes):
    for row in range(2):
        ax = axes[row, col]
        ax.set_facecolor('#0d0d0d')
        ax.imshow(np.rot90(flair_sl), cmap='gray', aspect='equal')
        if row == 1: ax.imshow(np.rot90(seg_rgba(seg_sl)), aspect='equal')
        ax.set_title(f'{title} {"FLAIR" if row==0 else "+ Segmentation"}',
                     color='white', fontsize=11, fontweight='bold')
        ax.axis('off')

legend_patches = [
    mpatches.Patch(color='red',    label='Tumor Core (NCR+ET)'),
    mpatches.Patch(color='yellow', label='Peritumoral Edema'),
    mpatches.Patch(color='lime',   label='Enhancing Tumor'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=12,
           facecolor='#1a1a1a', edgecolor='#444', labelcolor='white',
           framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
plt.tight_layout(rect=[0,0.04,1,0.98])

VIZ_PATH = f'/content/{PATIENT_ID}_segmentation_viz.png'
plt.savefig(VIZ_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
with open(VIZ_PATH,'rb') as _f: VIZ_B64 = base64.b64encode(_f.read()).decode()
print(f'✅ Static visualization saved -> {VIZ_PATH}')

# ── Generate slice stacks for interactive navigator ───────────────────────────
print('\nGenerating slice stacks for interactive dashboard navigator...')
N_SLICES = 24   # number of axial slices to export

D = flair_d.shape[2]
slice_indices = np.linspace(int(D*0.2), int(D*0.8), N_SLICES).astype(int)

AXIAL_FLAIR_B64  = []
AXIAL_SEG_B64    = []

for idx in slice_indices:
    AXIAL_FLAIR_B64.append(slice_to_b64(flair_d[:,:,idx]))
    AXIAL_SEG_B64.append(slice_to_b64(flair_d[:,:,idx], seg_d[:,:,idx]))

SLICE_CENTRE_IDX = int(N_SLICES // 2)  # default to middle slice

print(f'✅ {N_SLICES} axial slice pairs generated for navigator.')


## Cell 16 — Julich Atlas Regional Bar Chart

In [ ]:
region_names = [r['Region'][:40] + ('...' if len(r['Region'])>40 else '') for r in top5]
pct_tumor  = [r['Percentage_of_Tumor']           for r in top5]
pct_region = [r['Percentage_of_Region_Affected'] for r in top5]

fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
fig2.patch.set_facecolor('#111')
for ax in (ax1, ax2): ax.set_facecolor('#1a1a1a')

y = np.arange(len(region_names))[::-1]

ax1.barh(y, pct_tumor,  color='#667eea', edgecolor='none', height=0.55)
ax1.set_yticks(y); ax1.set_yticklabels(region_names, color='white', fontsize=9)
ax1.set_xlabel('% of Total Tumor Mass', color='#ccc', fontsize=11)
ax1.set_title('Tumor Distribution', color='white', fontsize=13, fontweight='bold')
ax1.tick_params(colors='#888')
for sp in ax1.spines.values(): sp.set_edgecolor('#333')
for i,v in enumerate(pct_tumor): ax1.text(v+.3, y[i], f'{v:.1f}%', va='center', color='white', fontsize=9)

ax2.barh(y, pct_region, color='#48bb78', edgecolor='none', height=0.55)
ax2.set_yticks(y); ax2.set_yticklabels(region_names, color='white', fontsize=9)
ax2.set_xlabel('% of Region Volume Affected', color='#ccc', fontsize=11)
ax2.set_title('Regional Involvement', color='white', fontsize=13, fontweight='bold')
ax2.tick_params(colors='#888')
for sp in ax2.spines.values(): sp.set_edgecolor('#333')
for i,v in enumerate(pct_region): ax2.text(v+.1, y[i], f'{v:.1f}%', va='center', color='white', fontsize=9)

fig2.suptitle(f'Top-5 Julich Atlas Regions | Patient: {PATIENT_ID}', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
CHART_PATH = f'/content/{PATIENT_ID}_atlas_chart.png'
plt.savefig(CHART_PATH, dpi=150, bbox_inches='tight', facecolor=fig2.get_facecolor())
plt.show()
with open(CHART_PATH, 'rb') as _f: CHART_B64 = base64.b64encode(_f.read()).decode()
print(f'✅ Chart saved -> {CHART_PATH}')


## Cell 17 — AI Clinical Report Generation

**Primary:** BioMistral-7B via HuggingFace Inference API  
**Fallback:** Rule-based structured 7-section template  

Optionally set `HF_TOKEN` for faster model loading.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 17 — AI Clinical Report Generation
#
# Primary:  HuggingFace BioMistral-7B Inference API
# Fallback: Rule-based structured template
# ─────────────────────────────────────────────────────────────────────────────
import requests

# ─── ⚙️ Optional: set your HuggingFace token for faster / gated model access ─
HF_TOKEN = ''   # leave blank for public access (may be slower)
# ─────────────────────────────────────────────────────────────────────────────

def build_prompt(meta, top5_regions, vols):
    region_lines = '\n'.join(
        f'  {i+1}. {r["Region"]}: '
        f'{r["Percentage_of_Tumor"]:.1f}% of tumor core, '
        f'{r["Percentage_of_Region_Affected"]:.2f}% of that region affected'
        for i, r in enumerate(top5_regions)
    )
    return (
        'You are a board-certified neuroradiologist writing a formal clinical MRI report. '
        'Generate a comprehensive, structured brain tumor analysis report based on the '
        'AI-assisted segmentation findings below. Use precise medical terminology.\n\n'
        'PATIENT INFORMATION\n'
        f'Patient ID            : {meta["patient_id"]}\n'
        f'Age / Sex             : {meta["age"]} years / {meta["sex"]}\n'
        f'Referring Physician   : {meta.get("referring_physician","N/A")}\n'
        f'Institution           : {meta.get("institution","N/A")}\n'
        f'Scan Date             : {meta["scan_date"]}\n'
        f'Clinical Indication   : {meta.get("clinical_indication","N/A")}\n'
        f'Presenting Symptoms   : {meta.get("presenting_symptoms","N/A")}\n'
        f'KPS Score             : {meta.get("kps_score","N/A")}\n\n'
        'AI SEGMENTATION FINDINGS  (AGSE-VNet, Epoch-55)\n'
        f'Whole Tumor Volume    : {vols["Whole_Tumor"]:.3f} cm3\n'
        f'Tumor Core Volume     : {vols["Tumor_Core"]:.3f} cm3 (NCR + ET)\n'
        f'Enhancing Tumor Volume: {vols["Enhancing_Tumor"]:.3f} cm3\n\n'
        'TOP-5 JULICH BRAIN ATLAS REGIONS (Tumor Core)\n'
        f'{region_lines}\n\n'
        'Write a structured clinical report with these sections:\n'
        'SECTION 1: CLINICAL SUMMARY\n'
        'SECTION 2: TECHNIQUE\n'
        'SECTION 3: IMAGING FINDINGS\n'
        'SECTION 4: TUMOR LOCALIZATION & FUNCTIONAL IMPLICATIONS\n'
        'SECTION 5: VOLUMETRIC ANALYSIS\n'
        'SECTION 6: IMPRESSION\n'
        'SECTION 7: RECOMMENDATIONS\n\n'
        'End with: "DISCLAIMER: This report is AI-assisted (AGSE-VNet + Julich Atlas) '
        'and must be reviewed and countersigned by a qualified neuroradiologist."\n'
    )


def generate_via_biomistral(prompt, hf_token=''):
    url     = 'https://api-inference.huggingface.co/models/BioMistral/BioMistral-7B'
    headers = {'Content-Type': 'application/json'}
    if hf_token:
        headers['Authorization'] = f'Bearer {hf_token}'
    resp = requests.post(url, headers=headers, json={
        'inputs': prompt,
        'parameters': {
            'max_new_tokens'  : 1200,
            'temperature'     : 0.35,
            'do_sample'       : True,
            'return_full_text': False,
        }
    }, timeout=120)
    if resp.status_code != 200:
        raise ValueError(f'HF API {resp.status_code}: {resp.text[:300]}')
    result = resp.json()
    if isinstance(result, list):
        return result[0].get('generated_text', '').strip()
    return result.get('generated_text', '').strip()


def generate_template_report(meta, top5, vols):
    region_lines = '\n'.join(
        f'  {i+1}. {r["Region"]}\n'
        f'       Tumor burden  : {r["Percentage_of_Tumor"]:.2f}% of total tumor core\n'
        f'       Region effect : {r["Percentage_of_Region_Affected"]:.3f}% of region volume affected'
        for i, r in enumerate(top5)
    )
    r0 = top5[0] if top5 else {}
    tc_wt_ratio = (vols['Tumor_Core']     / max(vols['Whole_Tumor'], 0.001)) * 100
    et_tc_ratio = (vols['Enhancing_Tumor'] / max(vols['Tumor_Core'],  0.001)) * 100
    ed_vol      = round(vols['Whole_Tumor'] - vols['Tumor_Core'], 3)

    return (
        'SECTION 1: CLINICAL SUMMARY\n'
        f'{meta.get("clinical_indication","Evaluation of intracranial mass")} in a '
        f'{meta["age"]}-year-old {meta["sex"].lower()} patient presenting with '
        f'{meta.get("presenting_symptoms","clinical symptoms")}. '
        f'AI-assisted multi-parametric MRI segmentation performed. '
        f'KPS score: {meta.get("kps_score","not recorded")}.\n\n'

        'SECTION 2: TECHNIQUE\n'
        'Multi-parametric MRI segmentation using AGSE-VNet (3D U-Net + Attention Gates + '
        'SE Blocks, Epoch-55, BraTS 2021). Modalities: T1, T1CE (gadolinium), T2, FLAIR. '
        'Segmentation registered to MNI152 standard space via ANTsPy SyN diffeomorphic '
        'registration. Tumor core voxels mapped to Julich Brain Cytoarchitectonic Atlas '
        '(siibra-python v3.1).\n\n'

        'SECTION 3: IMAGING FINDINGS\n'
        'The AI segmentation identifies the following tumor compartments:\n'
        f'  Whole Tumor (WT)      : {vols["Whole_Tumor"]:.2f} cm3  (NCR + Edema + ET)\n'
        f'  Tumor Core (TC)       : {vols["Tumor_Core"]:.2f} cm3  (NCR + ET)\n'
        f'  GD-Enhancing Tumor    : {vols["Enhancing_Tumor"]:.2f} cm3\n'
        f'  Peritumoral Edema     : {ed_vol:.2f} cm3\n'
        'Signal characteristics follow BraTS multi-parametric MRI conventions. '
        'The enhancing component demonstrates blood-brain barrier disruption on T1CE.\n\n'

        'SECTION 4: TUMOR LOCALIZATION & FUNCTIONAL IMPLICATIONS\n'
        f'Julich Brain Cytoarchitectonic Atlas mapping identifies {len(top5)} principal '
        f'anatomical regions involved by tumor core:\n\n'
        f'{region_lines}\n\n'
        f'The primary region of involvement is {r0.get("Region","the identified region")}, '
        f'accounting for {r0.get("Percentage_of_Tumor",0):.1f}% of the tumor core mass '
        f'({r0.get("Percentage_of_Region_Affected",0):.2f}% of that region affected). '
        'The spatial distribution requires assessment for eloquent cortex and white matter '
        'tract involvement prior to surgical planning.\n\n'

        'SECTION 5: VOLUMETRIC ANALYSIS\n'
        f'Tumor core-to-whole tumor ratio : {tc_wt_ratio:.1f}%  — '
        f'{"high necrotic burden" if tc_wt_ratio > 60 else "moderate necrotic component"}.\n'
        f'Enhancing fraction of tumor core : {et_tc_ratio:.1f}%  — '
        f'{"substantial active enhancement, aggressive biology" if et_tc_ratio > 40 else "moderate enhancement"}.\n'
        f'Total lesion volume {vols["Whole_Tumor"]:.2f} cm3 '
        f'{"exceeds" if vols["Whole_Tumor"] > 50 else "is within"} threshold for '
        'significant mass effect on surrounding structures.\n\n'

        'SECTION 6: IMPRESSION\n'
        f'1. Multi-compartment intracranial mass; total volume {vols["Whole_Tumor"]:.2f} cm3.\n'
        f'2. Tumor core {vols["Tumor_Core"]:.2f} cm3 with GD-enhancing component '
        f'({vols["Enhancing_Tumor"]:.2f} cm3) — most consistent with high-grade glioma '
        f'(WHO Grade III–IV); GBM cannot be excluded.\n'
        f'3. Primary involvement: {r0.get("Region","identified region")} per Julich Atlas.\n'
        f'4. KPS {meta.get("kps_score","N/A")} to be incorporated into treatment planning.\n\n'

        'SECTION 7: RECOMMENDATIONS\n'
        '1. Urgent neurosurgical consultation for biopsy vs. maximal safe resection.\n'
        '2. fMRI language and motor mapping if eloquent cortex is at risk.\n'
        '3. DTI tractography to assess corticospinal and arcuate fasciculus involvement.\n'
        '4. MR spectroscopy and perfusion imaging for grading correlation.\n'
        '5. Multidisciplinary neuro-oncology team review (neurosurgery, radiation oncology, '
        'medical oncology).\n'
        '6. Repeat MRI 6–8 weeks post-intervention per RANO criteria.\n\n'

        'DISCLAIMER: This report is AI-assisted (AGSE-VNet + Julich Atlas) '
        'and must be reviewed and countersigned by a qualified neuroradiologist.'
    )


# ── Run ───────────────────────────────────────────────────────────────────────
vols   = atlas_json['MRI_Scan']['Tumor_Details']['Volume_Estimates_cm3']
prompt = build_prompt(PATIENT_META, top5, vols)

REPORT_TEXT = ''
SOURCE_USED = ''

print('Calling BioMistral-7B via HuggingFace Inference API...')
print('(First call may take ~20–30s while the model loads on HF servers)\n')

try:
    REPORT_TEXT = generate_via_biomistral(prompt, HF_TOKEN)
    SOURCE_USED = 'BioMistral-7B (HuggingFace Inference API)'
    print('✅ Report generated via BioMistral-7B.')
except Exception as e:
    print(f'⚠️  BioMistral API unavailable: {e}')
    print('   Falling back to structured template report...\n')
    REPORT_TEXT = generate_template_report(PATIENT_META, top5, vols)
    SOURCE_USED = 'Rule-based structured template'
    print('✅ Template report generated.')

print(f'\nSource: {SOURCE_USED}')
print('\n' + '─'*70)
print(REPORT_TEXT)
print('─'*70)


## Cell 18 — Interactive Clinical Dashboard

Renders a **4-tab interactive dashboard**:
- 📊 **Overview** — volumetric stats + animated SVG risk gauge
- 🗺️ **Regional Analysis** — sortable, expandable atlas region cards
- 🖼️ **MRI Navigator** — 24-slice cine viewer + donut composition chart
- 📄 **Clinical Report** — formatted AI report + download buttons


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 18 — Interactive Clinical Dashboard
# ─────────────────────────────────────────────────────────────────────────────
import json as _json

primary_region_pct_tumor    = top5[0]['Percentage_of_Tumor']           if top5 else 0
primary_region_pct_affected = top5[0]['Percentage_of_Region_Affected'] if top5 else 0

dashboard_regions = [
    {
        'rank'         : i+1,
        'name'         : r['Region'],
        'tumor_voxels' : r['tc_voxels'],
        'pct_tumor'    : r['Percentage_of_Tumor'],
        'pct_region'   : r['Percentage_of_Region_Affected'],
    }
    for i, r in enumerate(top5)
]

vols     = atlas_json['MRI_Scan']['Tumor_Details']['Volume_Estimates_cm3']
wt_vol   = vols['Whole_Tumor']
tc_vol   = vols['Tumor_Core']
et_vol   = vols['Enhancing_Tumor']
ed_vol   = round(wt_vol - tc_vol, 3)
ncr_vol  = round(tc_vol - et_vol, 3)

# Risk score heuristic (0-100)
risk_score = min(100, int(
    (et_vol / max(wt_vol, 0.001)) * 50 +
    (tc_vol / max(wt_vol, 0.001)) * 30 +
    (len(region_voxel_count) / 30) * 20
))
risk_label = ('LOW' if risk_score < 35 else 'MODERATE' if risk_score < 65 else 'HIGH')
risk_color = ('#48bb78' if risk_score < 35 else '#ed8936' if risk_score < 65 else '#e53e3e')

region_json_str = _json.dumps(dashboard_regions)
report_json_str = _json.dumps(REPORT_TEXT)
slices_flair_js = _json.dumps(AXIAL_FLAIR_B64)
slices_seg_js   = _json.dumps(AXIAL_SEG_B64)
meta_json_str   = _json.dumps(PATIENT_META)

viz_tag   = ('<img src="data:image/png;base64,' + VIZ_B64 + '" style="max-width:100%;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.2)">'
             if 'VIZ_B64' in dir() else '<p style="color:#718096">Run Cell 15 first.</p>')
chart_tag = ('<img src="data:image/png;base64,' + CHART_B64 + '" style="max-width:100%;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.2);margin-top:20px">'
             if 'CHART_B64' in dir() else '')

# ── CSS ───────────────────────────────────────────────────────────────────────
CSS = """
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Segoe UI',system-ui,sans-serif;background:#f0f4f8;color:#2d3748;font-size:14px}

/* Header */
.hdr{background:linear-gradient(135deg,#0d47a1,#4a148c,#880e4f);color:#fff;padding:32px 40px;position:relative;overflow:hidden}
.hdr::after{content:'';position:absolute;top:-60px;right:-60px;width:300px;height:300px;
            background:rgba(255,255,255,.04);border-radius:50%}
.hdr h1{font-size:24px;font-weight:800;letter-spacing:-.3px}
.hdr .sub{font-size:13px;opacity:.8;margin-top:5px}
.hdr-meta{display:flex;flex-wrap:wrap;gap:10px;margin-top:14px}
.hdr-badge{background:rgba(255,255,255,.15);padding:6px 14px;border-radius:20px;font-size:13px;font-weight:600}

/* Tabs */
.tab-nav{display:flex;background:#1e293b;padding:0 24px;gap:2px;overflow-x:auto}
.tab-btn{padding:13px 20px;background:none;border:none;color:#94a3b8;cursor:pointer;
         font-size:13px;font-weight:600;border-bottom:3px solid transparent;
         transition:all .2s;white-space:nowrap}
.tab-btn.active,.tab-btn:hover{color:#fff;border-bottom-color:#818cf8}
.tab-content{display:none;padding:28px 36px;animation:fadeIn .3s ease}
.tab-content.active{display:block}
@keyframes fadeIn{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}

/* Stats Grid */
.stats-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:16px;margin-bottom:24px}
.stat-card{background:#fff;border-radius:14px;padding:20px;border-left:5px solid #818cf8;
           box-shadow:0 2px 10px rgba(0,0,0,.07);transition:transform .2s,box-shadow .2s;cursor:default}
.stat-card:hover{transform:translateY(-3px);box-shadow:0 6px 20px rgba(0,0,0,.12)}
.stat-label{font-size:10px;color:#94a3b8;text-transform:uppercase;font-weight:700;letter-spacing:.8px}
.stat-value{font-size:28px;font-weight:800;color:#1e293b;margin:8px 0 3px;line-height:1}
.stat-unit{font-size:11px;color:#cbd5e0}
.stat-icon{font-size:22px;margin-bottom:6px}

/* Risk Gauge */
.gauge-wrap{background:#fff;border-radius:14px;padding:24px;box-shadow:0 2px 10px rgba(0,0,0,.07);text-align:center}
.gauge-title{font-size:12px;color:#94a3b8;text-transform:uppercase;font-weight:700;letter-spacing:.8px;margin-bottom:16px}
.gauge-arc{margin:0 auto}

/* Region Cards */
.region-card{background:#fff;border-radius:14px;padding:22px;margin-bottom:14px;
             border:2px solid #e2e8f0;box-shadow:0 2px 8px rgba(0,0,0,.05);
             transition:border-color .2s,box-shadow .2s;cursor:pointer}
.region-card:hover,.region-card.selected{border-color:#818cf8;box-shadow:0 4px 20px rgba(129,140,248,.25)}
.rh{display:flex;align-items:center;gap:14px;margin-bottom:14px}
.rank{width:36px;height:36px;border-radius:50%;
      background:linear-gradient(135deg,#818cf8,#6366f1);
      color:#fff;display:flex;align-items:center;justify-content:center;
      font-weight:800;font-size:15px;flex-shrink:0}
.rname{flex:1;font-size:14px;font-weight:700;color:#1e293b}
.rvox{background:#f8fafc;padding:4px 12px;border-radius:16px;font-size:12px;font-weight:700;color:#64748b}
.bars{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.blabel{font-size:10px;color:#94a3b8;text-transform:uppercase;font-weight:700;margin-bottom:6px;letter-spacing:.5px}
.btrack{background:#e2e8f0;border-radius:10px;height:22px;overflow:hidden}
.bfill{height:100%;border-radius:10px;display:flex;align-items:center;padding-right:8px;
       justify-content:flex-end;transition:width 1.1s cubic-bezier(.4,0,.2,1)}
.bfill.blue{background:linear-gradient(90deg,#818cf8,#6366f1)}
.bfill.green{background:linear-gradient(90deg,#34d399,#059669)}
.btxt{color:#fff;font-size:11px;font-weight:800}
.region-detail{display:none;padding:14px;background:#f8fafc;border-radius:10px;margin-top:12px;font-size:13px;line-height:1.7;color:#475569}
.region-card.selected .region-detail{display:block}

/* Sort controls */
.sort-bar{display:flex;align-items:center;gap:8px;margin-bottom:18px;flex-wrap:wrap}
.sort-btn{padding:6px 14px;border:1px solid #e2e8f0;border-radius:8px;background:#fff;
          cursor:pointer;font-size:12px;font-weight:600;color:#64748b;transition:all .2s}
.sort-btn.active,.sort-btn:hover{background:#818cf8;color:#fff;border-color:#818cf8}

/* MRI Navigator */
.slice-nav{background:#0d0d0d;border-radius:14px;padding:20px;margin-bottom:20px}
.slice-nav-header{display:flex;justify-content:space-between;align-items:center;margin-bottom:14px}
.slice-nav-title{color:#fff;font-size:14px;font-weight:700}
.slice-toggle{display:flex;gap:8px}
.stbtn{padding:6px 14px;border:1px solid #444;border-radius:8px;background:#1a1a1a;
       color:#94a3b8;cursor:pointer;font-size:12px;font-weight:600;transition:all .2s}
.stbtn.active{background:#818cf8;color:#fff;border-color:#818cf8}
.slice-img-wrap{position:relative;text-align:center;min-height:300px}
.slice-img{max-width:100%;border-radius:8px;max-height:380px;object-fit:contain}
.slice-slider-wrap{margin-top:14px}
.slice-label{color:#94a3b8;font-size:12px;margin-bottom:6px;text-align:center}
input[type=range].slice-slider{
  width:100%;-webkit-appearance:none;appearance:none;height:6px;
  background:linear-gradient(90deg,#818cf8 var(--pct,50%),#333 var(--pct,50%));
  border-radius:3px;outline:none;cursor:pointer}
input[type=range].slice-slider::-webkit-slider-thumb{
  -webkit-appearance:none;width:18px;height:18px;border-radius:50%;
  background:#818cf8;cursor:pointer;box-shadow:0 2px 6px rgba(0,0,0,.4)}
.slice-info{display:flex;justify-content:space-between;color:#64748b;font-size:11px;margin-top:6px;padding:0 2px}
.play-btn{background:#818cf8;color:#fff;border:none;border-radius:8px;
          padding:8px 20px;font-weight:700;font-size:12px;cursor:pointer;margin-top:10px}

/* Donut Chart */
.donut-wrap{background:#fff;border-radius:14px;padding:24px;
            box-shadow:0 2px 10px rgba(0,0,0,.07);margin-bottom:20px}
.donut-title{font-size:13px;font-weight:700;color:#1e293b;margin-bottom:16px}
.donut-legend{display:flex;flex-wrap:wrap;gap:10px;margin-top:16px}
.legend-item{display:flex;align-items:center;gap:6px;font-size:12px;color:#475569}
.legend-dot{width:12px;height:12px;border-radius:50%;flex-shrink:0}
.donut-container{display:flex;align-items:center;gap:24px;flex-wrap:wrap}
.donut-center-text{text-align:center}
.donut-center-vol{font-size:28px;font-weight:800;color:#1e293b}
.donut-center-sub{font-size:11px;color:#94a3b8;margin-top:2px}

/* Report */
.report-wrap{background:#fff;border-radius:14px;padding:32px;
             box-shadow:0 2px 10px rgba(0,0,0,.07);line-height:1.85}
.report-source{display:inline-block;background:#f0fdf4;border:1px solid #bbf7d0;
               color:#166534;padding:4px 12px;border-radius:6px;font-size:11px;
               font-weight:700;margin-bottom:16px}
.report-section{margin-bottom:20px}
.report-section h3{font-size:15px;font-weight:800;color:#1e293b;
                   margin-bottom:8px;padding-bottom:6px;border-bottom:2px solid #e2e8f0}
.report-text{color:#374151;font-size:13px;line-height:1.8;white-space:pre-wrap}
.disclaimer-box{background:#fffbeb;border-left:4px solid #f59e0b;
                padding:14px 18px;border-radius:8px;margin-bottom:20px;
                font-size:13px;color:#78350f}
.disclaimer-box strong{display:block;margin-bottom:4px}

/* Print actions */
.action-bar{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px}
.act-btn{padding:10px 20px;border:none;border-radius:8px;cursor:pointer;
         font-weight:700;font-size:13px;display:flex;align-items:center;gap:6px}
.act-primary{background:#6366f1;color:#fff}
.act-secondary{background:#f1f5f9;color:#475569;border:1px solid #e2e8f0}
.act-btn:hover{opacity:.85}

/* Section title */
.sec-title{font-size:18px;font-weight:800;color:#1e293b;margin-bottom:18px;
           padding-bottom:10px;border-bottom:2px solid #e2e8f0}

/* Footer */
.footer{background:#1e293b;color:#94a3b8;text-align:center;
        padding:20px 40px;font-size:12px;line-height:1.7;margin-top:8px}

/* Tooltip */
.tooltip{position:relative}
.tooltip:hover::after{
  content:attr(data-tip);position:absolute;bottom:calc(100%+6px);left:50%;
  transform:translateX(-50%);background:#1e293b;color:#fff;padding:6px 12px;
  border-radius:6px;font-size:11px;white-space:nowrap;pointer-events:none;
  z-index:999}
"""

# ── JS ────────────────────────────────────────────────────────────────────────
JS_TEMPLATE = """
const regions      = REGION_JSON;
const reportText   = REPORT_JSON;
const metaData     = META_JSON;
const flairSlices  = FLAIR_SLICES_JSON;
const segSlices    = SEG_SLICES_JSON;

let currentSort  = 'pct_tumor';
let playInterval = null;
let showSeg      = true;
let sliceIdx     = Math.floor(flairSlices.length / 2);

// ── Tab switching ────────────────────────────────────────────────────────
function switchTab(i) {
  document.querySelectorAll('.tab-content').forEach((t,idx)=>t.classList.toggle('active',idx===i));
  document.querySelectorAll('.tab-btn').forEach((b,idx)=>b.classList.toggle('active',idx===i));
  if (i===1) setTimeout(animateBars,100);
  if (i===2) { setTimeout(()=>updateSlice(), 50); drawDonut(); }
  if (i===0) drawGauge();
}

// ── Gauge (SVG) ───────────────────────────────────────────────────────────
function drawGauge() {
  const score  = RISK_SCORE_VAL;
  const label  = 'RISK_LABEL_VAL';
  const color  = 'RISK_COLOR_VAL';
  const pct    = score / 100;
  const R      = 70, cx=90, cy=90;
  const start  = Math.PI, end = 2*Math.PI;
  const angle  = start + pct*(end-start);
  const x1=cx+R*Math.cos(start), y1=cy+R*Math.sin(start);
  const x2=cx+R*Math.cos(angle), y2=cy+R*Math.sin(angle);
  const large = pct > 0.5 ? 1 : 0;
  const svg = `<svg width="180" height="110" class="gauge-arc">
    <path d="M${x1},${y1} A${R},${R} 0 1,1 ${cx+R},${cy}" fill="none" stroke="#e2e8f0" stroke-width="16" stroke-linecap="round"/>
    <path d="M${x1},${y1} A${R},${R} 0 ${large},1 ${x2},${y2}" fill="none" stroke="${color}" stroke-width="16" stroke-linecap="round" style="transition:stroke-dashoffset 1s"/>
    <text x="${cx}" y="${cy-4}" text-anchor="middle" font-size="26" font-weight="800" fill="#1e293b">${score}</text>
    <text x="${cx}" y="${cy+18}" text-anchor="middle" font-size="13" font-weight="700" fill="${color}">${label} RISK</text>
    <text x="${cx}" y="${cy+34}" text-anchor="middle" font-size="10" fill="#94a3b8">/100</text>
  </svg>`;
  const el = document.getElementById('gauge-svg');
  if (el) el.innerHTML = svg;
}
drawGauge();

// ── Donut Chart (Canvas) ──────────────────────────────────────────────────
function drawDonut() {
  const canvas = document.getElementById('donut-canvas');
  if (!canvas) return;
  const ctx = canvas.getContext('2d');
  const cx=80, cy=80, R=65, r=40;
  const WT_VOL = WT_VOL_VAL;
  const segs = [
    { label:'Necrotic Core (NCR)', vol: NCR_VOL_VAL, color:'#f87171' },
    { label:'Enhancing Tumor (ET)', vol: ET_VOL_VAL, color:'#34d399' },
    { label:'Peritumoral Edema (ED)', vol: ED_VOL_VAL, color:'#fbbf24' },
  ];
  const total = segs.reduce((s,x)=>s+x.vol,0);
  let start = -Math.PI/2;
  ctx.clearRect(0,0,160,160);
  segs.forEach(seg=>{
    const sweep = (seg.vol/total)*2*Math.PI;
    ctx.beginPath();
    ctx.moveTo(cx,cy);
    ctx.arc(cx,cy,R,start,start+sweep);
    ctx.closePath();
    ctx.fillStyle=seg.color;
    ctx.fill();
    start+=sweep;
  });
  ctx.beginPath();
  ctx.arc(cx,cy,r,0,2*Math.PI);
  ctx.fillStyle='#fff';
  ctx.fill();
  ctx.fillStyle='#1e293b';
  ctx.font='bold 15px Segoe UI';
  ctx.textAlign='center';
  ctx.fillText(WT_VOL.toFixed(1),cx,cy+4);
  ctx.fillStyle='#94a3b8';
  ctx.font='9px Segoe UI';
  ctx.fillText('cm³ total',cx,cy+18);
}

// ── MRI Slice Navigator ────────────────────────────────────────────────────
function updateSlice() {
  const arr  = showSeg ? segSlices : flairSlices;
  const img  = document.getElementById('slice-img');
  const lbl  = document.getElementById('slice-lbl');
  const sl   = document.getElementById('slice-slider');
  if (!img || !arr[sliceIdx]) return;
  img.src = 'data:image/png;base64,' + arr[sliceIdx];
  lbl.textContent = 'Axial Slice ' + (sliceIdx+1) + ' / ' + arr.length;
  sl.value = sliceIdx;
  const pct = (sliceIdx / (arr.length-1)) * 100;
  sl.style.setProperty('--pct', pct+'%');
}

function setShowSeg(val) {
  showSeg = val;
  document.getElementById('btn-flair').classList.toggle('active', !val);
  document.getElementById('btn-seg').classList.toggle('active',  val);
  updateSlice();
}

function togglePlay() {
  const btn = document.getElementById('play-btn');
  if (playInterval) {
    clearInterval(playInterval); playInterval=null;
    btn.textContent = '▶ Play Cine';
  } else {
    playInterval = setInterval(()=>{
      sliceIdx = (sliceIdx+1) % flairSlices.length;
      updateSlice();
    }, 180);
    btn.textContent = '⏹ Stop';
  }
}

// ── Region Cards ──────────────────────────────────────────────────────────
function renderRegions(sortBy) {
  currentSort = sortBy;
  document.querySelectorAll('.sort-btn').forEach(b=>b.classList.toggle('active',b.dataset.sort===sortBy));
  const sorted = [...regions].sort((a,b)=>b[sortBy]-a[sortBy]);
  const list = document.getElementById('regions-list');
  list.innerHTML = '';
  sorted.forEach((r,i)=>{
    const card = document.createElement('div');
    card.className='region-card';
    card.id='rcard-'+i;
    card.innerHTML=`
      <div class="rh">
        <div class="rank">${r.rank}</div>
        <div class="rname">${r.name}</div>
        <div class="rvox">${r.tumor_voxels.toLocaleString()} voxels</div>
      </div>
      <div class="bars">
        <div>
          <div class="blabel">% of Total Tumor</div>
          <div class="btrack">
            <div class="bfill blue" style="width:0" data-w="${r.pct_tumor}%">
              <span class="btxt">${r.pct_tumor}%</span>
            </div>
          </div>
        </div>
        <div>
          <div class="blabel">% of Region Affected</div>
          <div class="btrack">
            <div class="bfill green" style="width:0" data-w="${r.pct_region}%">
              <span class="btxt">${r.pct_region}%</span>
            </div>
          </div>
        </div>
      </div>
      <div class="region-detail">
        <strong>Anatomical Region:</strong> ${r.name}<br>
        <strong>Tumor Core Voxels:</strong> ${r.tumor_voxels.toLocaleString()} voxels in MNI152 space<br>
        <strong>Tumor Burden:</strong> ${r.pct_tumor}% of total tumor core originates in this region<br>
        <strong>Regional Impact:</strong> ${r.pct_region}% of this brain region's volume is affected<br>
        <strong>Clinical Note:</strong> ${r.pct_region > 5 ? 'Significant regional involvement — assess functional eloquence.' : 'Partial regional involvement — monitor for progression.'}
      </div>`;
    card.onclick = ()=>{
      card.classList.toggle('selected');
    };
    list.appendChild(card);
  });
  setTimeout(animateBars, 80);
}

function animateBars() {
  document.querySelectorAll('.bfill').forEach(b=>b.style.width=b.dataset.w);
}

renderRegions('pct_tumor');

// ── Format report into sections ────────────────────────────────────────────
function formatReport(text) {
  const container = document.getElementById('report-content');
  if (!container) return;
  const sections = text.split(/(?=SECTION \\d+:)/);
  container.innerHTML = '';
  sections.forEach(sec => {
    if (!sec.trim()) return;
    const div = document.createElement('div');
    div.className = 'report-section';
    const lines = sec.trim().split('\\n');
    const title = lines[0];
    const body  = lines.slice(1).join('\\n').trim();
    if (title.match(/^SECTION \\d+:/)) {
      const h3 = document.createElement('h3');
      h3.textContent = title;
      div.appendChild(h3);
    }
    const p = document.createElement('div');
    p.className='report-text';
    p.textContent = body;
    div.appendChild(p);
    container.appendChild(div);
  });
  if (!sections.length || sections.every(s=>!s.match(/^SECTION/))) {
    const p = document.createElement('div');
    p.className='report-text';
    p.textContent = text;
    container.appendChild(p);
  }
}
formatReport(reportText);

// ── Download ───────────────────────────────────────────────────────────────
function downloadReport() {
  const header = [
    'CLINICAL BRAIN TUMOR ANALYSIS REPORT',
    '='.repeat(60),
    'Patient ID   : ' + metaData.patient_id,
    'Age / Sex    : ' + metaData.age + ' / ' + metaData.sex,
    'Scan Date    : ' + metaData.scan_date,
    'Indication   : ' + (metaData.clinical_indication || 'N/A'),
    '='.repeat(60),
    ''
  ].join('\\n');
  const blob = new Blob([header + reportText], {type:'text/plain'});
  const a = document.createElement('a');
  a.href = URL.createObjectURL(blob);
  a.download = 'ClinicalReport_' + metaData.patient_id + '.txt';
  a.click();
}

function downloadJSON() {
  const blob = new Blob([JSON.stringify({patient: metaData, regions: regions}, null, 2)],
                        {type:'application/json'});
  const a = document.createElement('a');
  a.href = URL.createObjectURL(blob);
  a.download = 'AtlasData_' + metaData.patient_id + '.json';
  a.click();
}
"""

# ── Build HTML ────────────────────────────────────────────────────────────────
def sc(label, value, unit, color='#818cf8', icon=''):
    return (f'<div class="stat-card tooltip" data-tip="{label}">'
            f'<div class="stat-icon">{icon}</div>'
            f'<div class="stat-label">{label}</div>'
            f'<div class="stat-value" style="color:{color}">{value}</div>'
            f'<div class="stat-unit">{unit}</div></div>')

stat_cards = (
    sc('Whole Tumor Volume',  f'{wt_vol:.2f}', 'cm³', '#818cf8', '🧠') +
    sc('Tumor Core (NCR+ET)', f'{tc_vol:.2f}', 'cm³', '#e53e3e', '🎯') +
    sc('Enhancing Tumor',     f'{et_vol:.2f}', 'cm³', '#34d399', '✨') +
    sc('Peritumoral Edema',   f'{ed_vol:.2f}', 'cm³', '#f59e0b', '💛') +
    sc('Regions Affected',    str(len(region_voxel_count)), 'Julich regions', '#38bdf8', '🗺️') +
    sc('TC Voxels (MNI)',     f'{TC_TOTAL_VOXELS:,}', 'voxels', '#a78bfa', '📍')
)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>AGSE-VNet Clinical Brain Tumor Report — {PATIENT_ID}</title>
<style>{CSS}</style>
</head>
<body>

<div class="hdr">
  <h1>🧠 Clinical Brain Tumor Analysis Dashboard</h1>
  <div class="sub">AGSE-VNet Epoch-55 · Julich Brain Cytoarchitectonic Atlas · AI-Assisted Report</div>
  <div class="hdr-meta">
    <span class="hdr-badge">👤 {PATIENT_META['patient_id']}</span>
    <span class="hdr-badge">🎂 {PATIENT_META['age']} yrs / {PATIENT_META['sex']}</span>
    <span class="hdr-badge">📅 {PATIENT_META['scan_date']}</span>
    <span class="hdr-badge">🏥 {PATIENT_META.get('institution','N/A')}</span>
    <span class="hdr-badge">⚕️ KPS {PATIENT_META.get('kps_score','N/A')}</span>
  </div>
</div>

<div class="tab-nav">
  <button class="tab-btn active" onclick="switchTab(0)">📊 Overview</button>
  <button class="tab-btn" onclick="switchTab(1)">🗺️ Regional Analysis</button>
  <button class="tab-btn" onclick="switchTab(2)">🖼️ MRI Navigator</button>
  <button class="tab-btn" onclick="switchTab(3)">📄 Clinical Report</button>
</div>

<!-- TAB 0: Overview -->
<div class="tab-content active" id="tab-0">
  <div style="display:grid;grid-template-columns:1fr 220px;gap:20px;align-items:start">
    <div>
      <div class="stats-grid">{stat_cards}</div>
      <div class="region-card" style="border-color:#818cf8">
        <h2 class="sec-title" style="font-size:15px">Primary Affected Region</h2>
        <div class="rh">
          <div class="rank">1</div>
          <div class="rname">{top5[0]['Region'] if top5 else 'N/A'}</div>
          <div class="rvox">{top5[0]['tc_voxels']:,} voxels</div>
        </div>
        <p style="font-size:13px;color:#64748b;line-height:1.7">
          Contains <strong>{primary_region_pct_tumor:.1f}%</strong> of the total tumor core, with
          <strong>{primary_region_pct_affected:.2f}%</strong> of the region's tissue volume involved.
          Clinical indication: <em>{PATIENT_META.get('clinical_indication','N/A')}</em>.
        </p>
      </div>
    </div>
    <div>
      <div class="gauge-wrap">
        <div class="gauge-title">Imaging Risk Score</div>
        <div id="gauge-svg"></div>
        <div style="font-size:11px;color:#94a3b8;margin-top:8px;line-height:1.5">
          Heuristic based on ET fraction,<br>TC ratio, and regions affected.
        </div>
      </div>
    </div>
  </div>
</div>

<!-- TAB 1: Regional Analysis -->
<div class="tab-content" id="tab-1">
  <h2 class="sec-title">Top Julich Brain Regions — Tumor Core Burden</h2>
  <div class="sort-bar">
    <span style="font-size:12px;color:#64748b;font-weight:600">Sort by:</span>
    <button class="sort-btn active" data-sort="pct_tumor"  onclick="renderRegions('pct_tumor')">% of Tumor</button>
    <button class="sort-btn"        data-sort="pct_region" onclick="renderRegions('pct_region')">% Region Affected</button>
    <button class="sort-btn"        data-sort="tumor_voxels" onclick="renderRegions('tumor_voxels')">Voxel Count</button>
    <span style="font-size:11px;color:#94a3b8;margin-left:8px">Click a card to expand details</span>
  </div>
  <div id="regions-list"></div>
</div>

<!-- TAB 2: MRI Navigator -->
<div class="tab-content" id="tab-2">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;align-items:start">

    <div>
      <h2 class="sec-title">Interactive MRI Slice Navigator</h2>
      <div class="slice-nav">
        <div class="slice-nav-header">
          <span class="slice-nav-title">Axial View — MNI152 Space</span>
          <div class="slice-toggle">
            <button class="stbtn" id="btn-flair" onclick="setShowSeg(false)">FLAIR</button>
            <button class="stbtn active" id="btn-seg" onclick="setShowSeg(true)">+ Segmentation</button>
          </div>
        </div>
        <div class="slice-img-wrap">
          <img id="slice-img" class="slice-img" src="" alt="MRI Slice">
        </div>
        <div class="slice-slider-wrap">
          <div class="slice-label" id="slice-lbl">Loading...</div>
          <input type="range" class="slice-slider" id="slice-slider"
                 min="0" max="{len(AXIAL_FLAIR_B64)-1}" value="{SLICE_CENTRE_IDX}"
                 oninput="sliceIdx=+this.value; updateSlice()">
          <div class="slice-info">
            <span>Superior</span>
            <span>Inferior</span>
          </div>
        </div>
        <div style="text-align:center;margin-top:8px">
          <button class="play-btn" id="play-btn" onclick="togglePlay()">▶ Play Cine</button>
        </div>
      </div>
    </div>

    <div>
      <h2 class="sec-title">Tumor Composition</h2>
      <div class="donut-wrap">
        <div class="donut-title">Volumetric Breakdown</div>
        <div class="donut-container">
          <canvas id="donut-canvas" width="160" height="160"></canvas>
          <div>
            <div class="donut-legend">
              <div class="legend-item"><div class="legend-dot" style="background:#f87171"></div>Necrotic Core — {ncr_vol:.2f} cm³</div>
              <div class="legend-item"><div class="legend-dot" style="background:#34d399"></div>Enhancing Tumor — {et_vol:.2f} cm³</div>
              <div class="legend-item"><div class="legend-dot" style="background:#fbbf24"></div>Peritumoral Edema — {ed_vol:.2f} cm³</div>
            </div>
            <div style="margin-top:16px;font-size:12px;color:#64748b;line-height:1.7">
              <strong>TC / WT ratio:</strong> {(tc_vol/max(wt_vol,0.001)*100):.1f}%<br>
              <strong>ET / TC ratio:</strong> {(et_vol/max(tc_vol,0.001)*100):.1f}%
            </div>
          </div>
        </div>
      </div>

      <h2 class="sec-title" style="margin-top:20px">Atlas Overlay</h2>
      {viz_tag}
      <div style="margin-top:16px">{chart_tag}</div>
    </div>

  </div>
</div>

<!-- TAB 3: Clinical Report -->
<div class="tab-content" id="tab-3">
  <div class="disclaimer-box">
    <strong>⚠️ Clinical Disclaimer</strong>
    This report is AI-assisted (AGSE-VNet + Julich Brain Atlas) and is intended to SUPPORT,
    not replace, clinical judgment. All findings must be reviewed and countersigned by a
    qualified neuroradiologist or neurosurgeon.
  </div>

  <div class="action-bar">
    <button class="act-btn act-primary" onclick="downloadReport()">💾 Download Report (.txt)</button>
    <button class="act-btn act-secondary" onclick="downloadJSON()">📊 Download Atlas JSON</button>
    <button class="act-btn act-secondary" onclick="window.print()">🖨️ Print Dashboard</button>
  </div>

  <div class="report-wrap">
    <div class="report-source">Generated by: {SOURCE_USED}</div>
    <div id="report-content"></div>
  </div>
</div>

<div class="footer">
  <strong>Technology Stack:</strong> AGSE-VNet 3D U-Net (Epoch 55) &middot;
  Julich Brain Cytoarchitectonic Atlas &middot; ANTsPy SyN Registration &middot;
  MNI152 2009c Nonlinear Space &middot; {SOURCE_USED}<br>
  <span style="opacity:.6;font-size:11px">Patient: {PATIENT_ID} &nbsp;|&nbsp; {PATIENT_META['scan_date']}</span>
</div>

<script>
{JS_TEMPLATE
  .replace('REGION_JSON',       region_json_str)
  .replace('REPORT_JSON',       report_json_str)
  .replace('META_JSON',         meta_json_str)
  .replace('FLAIR_SLICES_JSON', slices_flair_js)
  .replace('SEG_SLICES_JSON',   slices_seg_js)
  .replace('RISK_SCORE_VAL',    str(risk_score))
  .replace('RISK_LABEL_VAL',    risk_label)
  .replace('RISK_COLOR_VAL',    risk_color)
  .replace('WT_VOL_VAL',        str(wt_vol))
  .replace('ET_VOL_VAL',        str(et_vol))
  .replace('NCR_VOL_VAL',       str(ncr_vol))
  .replace('ED_VOL_VAL',        str(ed_vol))
}
updateSlice();
drawDonut();
</script>
</body></html>"""

dashboard_path = f'/content/{PATIENT_ID}_dashboard.html'
with open(dashboard_path, 'w', encoding='utf-8') as _f:
    _f.write(html)

display(HTML(html))
print(f'\n✅ Interactive dashboard rendered above.')
print(f'   HTML saved -> {dashboard_path}')


## Cell 19 — Save All Outputs to Google Drive

In [ ]:
import shutil

CASE_DIR = OUTPUT_DIR / PATIENT_ID
CASE_DIR.mkdir(parents=True, exist_ok=True)

files_to_save = {
    f'{PATIENT_ID}_atlas_mapping.json'    : f'/content/{PATIENT_ID}_atlas_mapping.json',
    f'{PATIENT_ID}_dashboard.html'        : f'/content/{PATIENT_ID}_dashboard.html',
    f'{PATIENT_ID}_segmentation.nii.gz'  : '/tmp/seg_pred_native.nii.gz',
    f'{PATIENT_ID}_tc_mask_mni.nii.gz'   : '/content/tc_mask_mni.nii.gz',
    f'{PATIENT_ID}_flair_mni.nii.gz'     : '/content/flair_mni.nii.gz',
    f'{PATIENT_ID}_segmentation_viz.png' : f'/content/{PATIENT_ID}_segmentation_viz.png',
    f'{PATIENT_ID}_atlas_chart.png'      : f'/content/{PATIENT_ID}_atlas_chart.png',
}

print(f'Saving outputs to: {CASE_DIR}')
for out_name, src_path in files_to_save.items():
    src = Path(src_path)
    if src.exists():
        shutil.copy2(src, CASE_DIR / out_name)
        sz = (CASE_DIR / out_name).stat().st_size / 1e6
        print(f'  ✅  {out_name}  ({sz:.2f} MB)')
    else:
        print(f'  ⚠️   Not found: {src_path}')

report_path = CASE_DIR / f'{PATIENT_ID}_clinical_report.txt'
with open(report_path, 'w') as _f:
    _f.write('CLINICAL BRAIN TUMOR ANALYSIS REPORT\n')
    _f.write('=' * 60 + '\n')
    _f.write(f'Patient: {PATIENT_ID}  |  {PATIENT_META["scan_date"]}\n')
    _f.write('=' * 60 + '\n\n')
    _f.write(REPORT_TEXT)
print(f'  ✅  {report_path.name}')
print(f'\n✅ All outputs saved -> {CASE_DIR}')
